In [1]:
import pandas as pd
import numpy as np
import joblib

# INCARCARE MODEL EXPORTAT
try:
    model_pack = joblib.load('models/profit_prediction_engine.pkl')
    print("✅ Modelul a fost incarcat cu succes.")
except FileNotFoundError:
    print("❌ Eroare: Fisierul .pkl nu a fost gasit. Verifica numele!")

# 2. CREARE DATE FICTIVE
new_data = pd.DataFrame([{
    'sales': 1500.0,
    'discount': 0.15,
    'quantity': 500,
    'is_high_season': 0,
    'unit_sales': 30.0,
    'risk_score': 1500.5,          # valoare medie din risk_lookup
    'discount_efficiency': 1.2,  # profit / (discount + 0.01) estimat
    'high_loss_risk': 0,
    'p_last': 0.5,              # marja de anul trecut
    'p_margin': 0.10             # marja curenta (pentru segmentare)
}])

print("\n--- TESTARE PRODUCTIE ---")

# SEGMENTARE
X_new_clust = model_pack['robust_scaler'].transform(new_data[['sales', 'p_margin', 'discount']])
segment = model_pack['kmeans_model'].predict(X_new_clust)[0]
print(f"Identificare Segment: {segment}")

# PREDICTIE MARJA
if segment in model_pack['experts']:
    expert_model = model_pack['experts'][segment]

    # predictia va fi in spatiul Yeo-Johnson (margin_transformed)
    features = model_pack['features_list']
    p_trans = expert_model.predict(new_data[features])


    p_df = pd.DataFrame(p_trans, columns=['p_margin']) # numele coloanei trebuie sa fie acelasi ca la fit
    final_margin = model_pack['power_transformer'].inverse_transform(p_df)


    predicted_profit = final_margin[0][0] * new_data['sales'].values[0]

    print(f"Marja de profit prezisa: {final_margin[0][0]*100:.2f}%")
    print(f"Predictie Profit Final: {predicted_profit:.2f}$")
else:
    print(f"Eroare: Nu exista un model antrenat pentru segmentul {segment}")

print("-" * 25)

❌ Eroare: Fisierul .pkl nu a fost gasit. Verifica numele!

--- TESTARE PRODUCTIE ---


NameError: name 'model_pack' is not defined